## RNNs in PyTorch

## Install PyTorch (CPU version)


In [2]:
!pip install https://download.pytorch.org/whl/cpu/torch-1.0.1.post2-cp36-cp36m-linux_x86_64.whl
!pip install torchvision

  Using cached https://download.pytorch.org/whl/cpu/torch-1.0.1.post2-cp36-cp36m-linux_x86_64.whl
ERROR: torchvision 0.5.0 has requirement torch==1.4.0, but you'll have torch 1.0.1.post2 which is incompatible.
  Found existing installation: torch 1.4.0
    Uninstalling torch-1.4.0:
      Successfully uninstalled torch-1.4.0
  Using cached https://files.pythonhosted.org/packages/24/19/4804aea17cd136f1705a5e98a00618cb8f6ccc375ad8bfa437408e09d058/torch-1.4.0-cp36-cp36m-manylinux1_x86_64.whl
  Found existing installation: torch 1.0.1.post2
    Uninstalling torch-1.0.1.post2:
      Successfully uninstalled torch-1.0.1.post2


## Verify PyTorch installation


In [1]:
import torch
print(torch.__version__)

1.4.0


## Download & unzip fasttext word embeddings

In [4]:
!wget https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.vec.gz
!gzip -d cc.en.300.vec.gz

--2020-03-04 12:36:46--  https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.vec.gz
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 104.20.22.166, 104.20.6.166, 2606:4700:10::6814:6a6, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|104.20.22.166|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1325960915 (1.2G) [binary/octet-stream]
Saving to: ‘cc.en.300.vec.gz’

cc.en.300.vec.gz    100%[===================>]   1.23G  12.2MB/s    in 1m 45s  

2020-03-04 12:38:33 (12.0 MB/s) - ‘cc.en.300.vec.gz’ saved [1325960915/1325960915]



## Load embeddings 

In [0]:
import pickle
import numpy as np

idx = 0
vocab = {}
with open("cc.en.300.vec", 'r', encoding="utf-8", newline='\n',errors='ignore') as f:
    for l in f:
        line = l.rstrip().split(' ')
        if idx == 0:
            vocab_size = int(line[0]) + 2
            dim = int(line[1])
            vecs = np.zeros(vocab_size*dim).reshape(vocab_size,dim)
            vocab["__PADDING__"] = 0
            vocab["__UNK__"] = 1
            idx = 2
        else:
            vocab[line[0]] = idx
            emb = np.array(line[1:]).astype(np.float)
            if (emb.shape[0] == dim):
                vecs[idx,:] = emb
                idx+=1
            else:
                continue

    pickle.dump(vocab,open("fasttext_voc",'wb'))
    np.save("fasttext.npy",vecs)

### Load embeddings np-array and mapping dictionary

In [0]:
import pickle
import numpy as np

fasttext_embed = np.load("fasttext.npy")
fasttext_word_to_index = pickle.load(open("fasttext_voc", 'rb'))

## Load and pre-process reuter's dataset

In [3]:
import nltk
nltk.download('reuters')
from nltk.corpus import reuters
from sklearn.preprocessing import MultiLabelBinarizer
from nltk.tokenize import TweetTokenizer
tweet_wt = TweetTokenizer()

mlb = MultiLabelBinarizer()
documents = reuters.fileids()
test = [d for d in documents if d.startswith('test/')]
train = [d for d in documents if d.startswith('training/')]
docs = {}
docs['train'] = [tweet_wt.tokenize(reuters.raw(doc_id)) for doc_id in train]
docs['test'] = [tweet_wt.tokenize(reuters.raw(doc_id)) for doc_id in test]
y = {'train': [], 'test': []}
y['train'] = mlb.fit_transform([reuters.categories(doc_id) for doc_id in train])
y['test'] = mlb.transform([reuters.categories(doc_id) for doc_id in test])

[nltk_data] Downloading package reuters to /root/nltk_data...
[nltk_data]   Package reuters is already up-to-date!


### Function that calculates evaluation metrics (weighted macro f1, precision, recall and accuracy)

In [0]:
def metrics(output,target):

    eps = 1e-07 
    output = output.cpu().detach().numpy()
    target = target.cpu().numpy()
    true_positives = np.sum(np.round(np.clip(target * output, 0, 1)))
    possible_positives = np.sum(np.round(np.clip(target, 0, 1)))
    recall = true_positives / (possible_positives + eps)
    predicted_positives = np.sum(np.round(np.clip(output, 0, 1)))
    precision = true_positives / (predicted_positives + eps)
    f1 = 2 * ((precision * recall) / (precision + recall))
    acc = np.mean(target == (output >=0.5))
    
    
    return f1,precision,recall,acc

### Create Dataloader class

In [0]:
import torch
import pandas as pd
import numpy as np
import math
import random

class TextClassDataLoader(object):

    def __init__(self,data,word_to_index,labels=None,batch_size=32,predict_flag=0,train=0):
        """

        Args:
            data:
            word_to_index:
            labels:
            batch_size:
        """

        self.batch_size = batch_size
        self.word_to_index = word_to_index
        self.predict_flag = predict_flag
        self.train = train
        print("Train flag: ",self.train)
        print("Predict flag: ",self.predict_flag)
        self.data = data
        self.data_id  = self.indexifyer(self.data)
        self.labels = labels
        if self.labels is None:
            self.samples = self.data_id
        else:
            self.labels = labels.tolist()
            self.samples = list(zip(self.data_id,self.labels))

        # for batch
        self.n_samples = len(self.samples)
        self.n_batches = math.ceil(self.n_samples / self.batch_size)
        self.max_length = self._get_max_length()
        self.index = 0
        self.batch_index = 0
        self.indices = np.arange(self.n_samples)
        if self.train:
            self._shuffle_indices()

        self.report()

    def _shuffle_indices(self):
        self.indices = np.random.permutation(self.n_samples)
        self.index = 0
        self.batch_index = 0

    def _get_max_length(self):
        length = 0
        for sample in self.samples:
            length = max(length, len(sample[0]))
        return length

    def indexifyer(self,data):

        indexes = []
        for sent in data:
            sent_indices = []
            for tok in sent:
                if tok.lower() in self.word_to_index:
                    sent_indices.append(self.word_to_index[tok.lower()])
                else:
                    sent_indices.append(self.word_to_index['__UNK__'])
            indexes.append(sent_indices)

        return indexes

    def _create_batch(self):

        batch = []
        n = 0
        while ((n < self.batch_size) and (self.index < len(self.samples))):
            _index = self.indices[self.index]
            batch.append(self.samples[_index])
            self.index += 1
            n += 1
        self.batch_index += 1

        #Fix for the extreme case that last batch has size == 1. Append the last sample to the
        #previous batch
        if (self.index+1 == len(self.samples)):
            _index = self.indices[self.index]
            batch.append(self.samples[_index])
            self.index += 1
            self.batch_index += 1

        sentences = []
        labels = []

        for bat in batch:
            if self.labels is not None:
                labels.append(bat[1])
            indices = []
            for sent_ids in bat[0]:
                indices.append(sent_ids)
            sentences.append(indices)
        seq_lengths = torch.LongTensor(list(map(len, sentences)))


        #padding
        seq_tensor = torch.zeros((len(sentences), seq_lengths.max())).long()
        for idx, (seq, seqlen) in enumerate(zip(sentences, seq_lengths)):
            seq_tensor[idx, :seqlen] = torch.LongTensor(seq)


        #sort in decreasing order
        seq_lengths, perm_idx = seq_lengths.sort(0, descending=True)
        seq_tensor = seq_tensor[perm_idx]


        if self.labels is not None:
            labels = torch.FloatTensor(labels)
            labels = labels[perm_idx]
        else:
            labels = None

        return seq_tensor, labels, seq_lengths

    def __len__(self):
        return self.n_batches

    def __iter__(self):

        if self.train:
            self._shuffle_indices()
        else:
            self.index = 0
            self.batch_index = 0

        for i in range(self.n_batches):
            if self.batch_index == self.n_batches:
                raise StopIteration()
            yield self._create_batch()

    def show_samples(self, n=10):
        for sample in self.samples[:n]:
            print(sample)

    def report(self):
        print('# samples: {}'.format(len(self.samples)))
        print('max len: {}'.format(self.max_length))
        print('# vocab: {}'.format(len(self.word_to_index)))
        print('# batches: {} (batch_size = {})'.format(self.n_batches, self.batch_size))

### Create RNN class 
1. Select RNN type (either GRU or LSTM)
2. Define RNNs depth (num of layers) and size
3. Use either linear attention over all timestep outputs  or  get the last hidden state as sentence representation
4. Get one probability (logit) per class

In [0]:
import torch
import numpy as np
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import torch.nn.functional as F
from torch.autograd import Variable

class RNN(nn.Module):
    
    def __init__(self, vocab_size, embed_size, num_output, rnn_model='GRU', reccurent_size=128, embedding_tensor=None,
                 padding_index=0, num_layers=1, bidirectional=True, attention_flag=False, dropout=0):

        '''
        Args:
            vocab_size: vocab size
            embed_size: embeddings dim
            num_output: number of output (classes)
            rnn_model:  LSTM or GRU
            hidden_size: reccurent size
            embedding_tensor: Pretrainned embbedings tensor
            padding_index: index for padding
            hidden_size: hidden size of rnn module
            num_layers:  number of layers in rnn module
            attention_flag: whether to use linear attention or not
            dropout: dropout probability
        '''

        
        super(RNN, self).__init__()
      
        self.hidden = reccurent_size
        self.dropout = dropout
        self.num_output = num_output
        self.num_layers = num_layers
        self.attention_flag = attention_flag
        self.bidirectional = bidirectional
        self.batch_first = True
        
        self.encoder = nn.Embedding(vocab_size, embed_size, padding_idx=padding_index, _weight=embedding_tensor)
        self.encoder.weight.requires_grad = False
        self.dropout = self.dropout
        self.drop_en = nn.Dropout(self.dropout)


        #rnn module
        if rnn_model == 'LSTM':
            self.rnn = nn.LSTM(input_size=embed_size, hidden_size=self.hidden, num_layers=self.num_layers, dropout=0,
                                batch_first=self.batch_first, bidirectional=self.bidirectional)
        elif rnn_model == 'GRU':
            self.rnn = nn.GRU(input_size=embed_size, hidden_size=self.hidden, num_layers=self.num_layers, dropout=0,
                                batch_first=self.batch_first, bidirectional=self.bidirectional)
        else:
            raise LookupError('only support LSTM and GRU')


        if self.attention_flag:
            if self.bidirectional:
                self.att_weights = nn.Linear(self.hidden*2,1,bias=True)
                nn.init.xavier_uniform_(self.att_weights.weight.data)
                self.fc = nn.Linear(self.hidden*2, self.num_output)
            else:
                self.att_weights = nn.Linear(self.hidden,1,bias=True)
                nn.init.xavier_uniform_(self.att_weights.weight.data)
                self.fc = nn.Linear(self.hidden, self.num_output)
        else:
            if self.bidirectional:
                self.fc = nn.Linear(self.hidden*2, self.num_output)
            else:
                self.fc = nn.Linear(self.hidden, self.num_output)
            
            
    
    def forward(self,x,seq_lengths):
        
        #Lookup on embeddings matrix
        x_embed = self.encoder(x)
        x_embed = self.drop_en(x_embed)

        packed_input = pack_padded_sequence(x_embed, seq_lengths.cpu().numpy(),batch_first=True)
        packed_output, ht = self.rnn(packed_input, None)
        out_rnn, lengths = pad_packed_sequence(packed_output, batch_first=True)
        out_rnn = self.drop_en(out_rnn)

        #Either use linear-attention over all timesteps or get the last hidden state
        #as sentence representation
        if self.attention_flag:
            
            # apply attention layer
            weights = self.att_weights(out_rnn)

            # create mask based on the sentence lengths
            mask = torch.ones(weights.size())

            for i, l in enumerate(lengths):  # skip the first sentence
                if l < out_rnn.size()[1]: 
                    mask[i, l:] = 0

            #zero padded elements in softmax    
            weights = weights.masked_fill(mask == 0, -1e9)

            #apply attention and get sentence representations
            attentions = F.softmax(weights.squeeze(),dim=1)
            weighted = torch.mul(out_rnn, attentions.unsqueeze(-1).expand_as(out_rnn))
            representations = weighted.sum(1).squeeze()
        
        else:
            row_indices = torch.arange(0, x.size(0)).long()
            col_indices = seq_lengths - 1
            representations = out_rnn[row_indices, col_indices, :]

        out = self.fc(representations).squeeze(1)

        return out

### Create train and dev dataloaders

In [7]:
batch_size = 32
train_loader = TextClassDataLoader(docs["train"],fasttext_word_to_index,y['train'],batch_size,predict_flag=0,train=1)
print()
val_loader = TextClassDataLoader(docs["test"],fasttext_word_to_index,y['test'],batch_size,predict_flag=0,train=0)

Train flag:  1
Predict flag:  0
# samples: 7769
max len: 1490
# vocab: 2000002
# batches: 243 (batch_size = 32)

Train flag:  0
Predict flag:  0
# samples: 3019
max len: 2166
# vocab: 2000002
# batches: 95 (batch_size = 32)


In [8]:
train_loader.show_samples(2)

([389764, 14170, 574, 10385, 1552, 977, 3, 291, 10, 3, 389764, 14170, 3431, 2, 52995, 3, 11908, 232, 486, 43449, 5, 4200, 7850, 14, 3, 666, 1, 2, 986, 1341, 11393, 1149, 34, 38, 76, 6115, 2, 1, 30430, 116, 10, 104, 3580, 574, 4, 3, 1792, 812, 483, 3, 1, 46, 30, 774, 33, 153, 4, 21786, 14, 3, 291, 1609, 51013, 397, 66, 1, 3608, 7, 1410, 44555, 357, 8, 16534, 781, 14, 3, 424, 7, 224065, 81151, 280, 225867, 27, 3, 193, 1317, 180, 153, 4, 260, 21, 563, 15, 14170, 2675, 1312, 19, 36343, 24, 654, 10, 3, 21786, 2188, 4, 1, 30430, 116, 84, 11, 179, 87, 2025, 25, 6, 113, 139, 311, 6512, 14170, 11, 179, 285, 25, 18159, 49, 7799, 257, 6, 40, 273, 4, 20, 781, 389764, 6512, 4831, 194, 24652, 81151, 3608, 5, 1045, 2371, 27, 566, 22656, 81151, 84, 26, 8, 213, 3306, 4136, 3608, 179, 10, 3, 1170, 7, 4962, 2, 68412, 2, 33022, 5, 12630, 4, 84, 26, 10823, 25, 6, 113, 139, 7, 33, 14170, 73, 30, 1094, 14, 7080, 25, 50396, 26, 129, 6365, 697096, 10, 8762, 245, 389764, 4387, 245, 10503, 4, 10, 513, 7, 3, 1046

### Load pretrained embeddings and create an RNN model

In [0]:
embeds = torch.FloatTensor(fasttext_embed)

In [10]:
embeds.size()

torch.Size([2000002, 300])

In [0]:
model = RNN(embeds.size()[0],embeds.size()[1], num_output=y["train"].shape[1], rnn_model='LSTM', reccurent_size=128, embedding_tensor=embeds,
                 padding_index=0, num_layers=1,bidirectional=True, attention_flag=True, dropout=0.2)

In [12]:
print(model)

RNN(
  (encoder): Embedding(2000002, 300, padding_idx=0)
  (drop_en): Dropout(p=0.2, inplace=False)
  (rnn): LSTM(300, 128, batch_first=True, bidirectional=True)
  (att_weights): Linear(in_features=256, out_features=1, bias=True)
  (fc): Linear(in_features=256, out_features=90, bias=True)
)


# Training 

In [0]:
from tqdm import tqdm_notebook as tqdm

#define optimizer and loss
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)
criterion = nn.BCEWithLogitsLoss()

#Train for 3 epochs and validate at the end of each epoch
#Keep the model with the best validation f1-score
epochs = 3
highest_val_f1 = 0
for idx in tqdm(range(epochs),desc="Epoch"):
    epoch = idx+1
    #Switch to train mode
    model.train() 
    for (input_t,target_t,lengths_t) in tqdm(train_loader, desc="Iteration"):
        output = model(input_t,lengths_t)
        loss = criterion(output,target_t)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    #Switch to eval mode
    model.eval()
    val_loss = []
    val_targets = []
    val_outputs = []
    for i, (input_t,target_t,lengths_t) in enumerate(val_loader):
        output = model(input_t,lengths_t)
        val_outputs.append(output)
        val_targets.append(target_t)
        val_loss.append(criterion(output,target_t).detach().numpy())
    val_outputs = torch.cat(val_outputs,0)
    val_targets = torch.cat(val_targets,0)
    f1,pr,rc,acc = metrics(val_outputs, val_targets)
    print("val loss: %.4f, val f1: %.3f, val precision: %.3f, val recall: %.3f, val accuracy: %.3f" %((sum(val_loss)/len(val_loss)),f1,pr,rc,acc))
    if f1 > highest_val_f1:
        print("Save model....")
        torch.save({'model_state_dict': model.state_dict()}, "pytorch_model.bin")
        highest_val_f1 = f1

val loss: 0.0427, val f1: 0.400, val precision: 0.981, val recall: 0.251, val accuracy: 0.990
Save model....


val loss: 0.0387, val f1: 0.442, val precision: 0.947, val recall: 0.288, val accuracy: 0.990
Save model....


val loss: 0.0371, val f1: 0.547, val precision: 0.900, val recall: 0.393, val accuracy: 0.991
Save model....



# Evaluation

In [0]:
#Define loss function
criterion = criterion = nn.BCEWithLogitsLoss()

#Create test dataloader
test_loader = TextClassDataLoader(docs["test"],fasttext_word_to_index,y['test'],batch_size,predict_flag=0,train=0)

#Create model instance using identical parameters with the already trained model
model_eval = RNN(embeds.size()[0],embeds.size()[1], num_output=y["train"].shape[1], rnn_model='LSTM', reccurent_size=128, embedding_tensor=embeds,
                 padding_index=0, num_layers=1,bidirectional=True, attention_flag=True, dropout=0.2)

#Load pre-trained model
print("===> Loading pretrained model ...")
checkpoint = torch.load("pytorch_model.bin", map_location="cpu")
model_eval.load_state_dict(checkpoint['model_state_dict'])

#Switch to evaluation mode
model_eval.eval()
val_loss = []
val_targets = []
val_outputs = []

for i, (input_t, target_t,lengths_t) in enumerate(val_loader):
    output = model_eval(input_t,lengths_t)
    val_outputs.append(output)
    val_targets.append(target_t)
    val_loss.append(criterion(output,target_t).detach().numpy())
val_outputs = torch.cat(val_outputs,0)
val_targets = torch.cat(val_targets,0)
f1,pr,rc,acc = metrics(val_outputs, val_targets)
print("val loss: %.4f, val f1: %.3f, val precision: %.3f, val recall: %.3f, val accuracy: %.3f" %((sum(val_loss)/len(val_loss)),f1,pr,rc,acc))

Train flag:  0
Predict flag:  0
# samples: 3019
max len: 2166
# vocab: 2000002
# batches: 95 (batch_size = 32)
===> Loading pretrained model ...
val loss: 0.0371, val f1: 0.547, val precision: 0.900, val recall: 0.393, val accuracy: 0.991
